# Segmentación de Países

Proyecto de clustering listo para **Google Colab**.

**Pregunta:** ¿podemos agrupar países según indicadores socioeconómicos?

**Método:** K-Means.

**Stack:** pandas, seaborn, scikit-learn

Dataset de ejemplo para segmentación de países.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar / simular datos

In [ ]:
np.random.seed(RANDOM_STATE)
n = 100
df = pd.DataFrame({
    "GDP_per_capita": np.random.uniform(500, 80000, n).round(0),
    "Life_expectancy": np.random.uniform(50, 85, n).round(1),
    "Infant_mortality": np.random.uniform(2, 80, n).round(1),
    "Literacy_rate": np.random.uniform(40, 100, n).round(1),
})
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df.describe().T)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df["GDP_per_capita"], kde=True, ax=axes[0,0])
sns.histplot(df["Life_expectancy"], kde=True, ax=axes[0,1])
sns.histplot(df["Infant_mortality"], kde=True, ax=axes[1,0])
sns.histplot(df["Literacy_rate"], kde=True, ax=axes[1,1])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.scatterplot(data=df, x="GDP_per_capita", y="Life_expectancy")
plt.title("GDP vs Esperanza de vida")
plt.show()

### Hallazgos
- Correlaciones esperadas entre indicadores.
- Ideal para clustering de países.

## 3. Preprocesamiento

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
print(X_scaled.shape)

## 4. Elegir k (Elbow + Silhouette)

In [ ]:
inertias = []
silhouettes = []
K = range(2, 8)

for k in K:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K, inertias, marker="o")
axes[0].set_title("Elbow")
axes[1].plot(K, silhouettes, marker="o", color="green")
axes[1].set_title("Silhouette")
plt.tight_layout()
plt.show()

## 5. Clustering final

In [ ]:
k_final = 3
km = KMeans(n_clusters=k_final, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = km.fit_predict(X_scaled)
print(df["cluster"].value_counts())

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="GDP_per_capita", y="Life_expectancy", hue="cluster", palette="tab10")
plt.title("Clusters de países")
plt.show()

## 6. Perfil de segmentos

In [ ]:
perfil = df.groupby("cluster").mean().round(1)
perfil["n"] = df["cluster"].value_counts()
perfil

## 7. Conclusiones

- Segmentos típicos: desarrollados, emergentes, en desarrollo.
- k=3 suele capturar bien los grupos.
- Se puede extender a datasets reales de World Bank o UN.

**Siguiente paso:** nombrar los clusters según indicadores.